In [ ]:
import os
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from kneed import KneeLocator
import matplotlib.pyplot as plt
import joblib
import os, gc, time, glob
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pickle

In [ ]:
# RoBERTa embedding

from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding

tokenizer = RobertaTokenizerFast.from_pretrained("entropy/roberta_zinc_480m", max_len=128)
model_emb = RobertaForMaskedLM.from_pretrained('entropy/roberta_zinc_480m')
collator = DataCollatorWithPadding(tokenizer, padding=True, return_tensors='pt')

def Create_embedding(smiles):
    inputs = collator(tokenizer(smiles))
    outputs = model_emb(**inputs, output_hidden_states=True)
    full_embeddings = outputs[1][-1]
    mask = inputs['attention_mask']
    embeddings = ((full_embeddings * mask.unsqueeze(-1)).sum(1) / mask.sum(-1).unsqueeze(-1))
    return embeddings

In [ ]:
# output directory

ssd_dir = '../data/SelfDiff_SSD_20251117_threshold1'
output_dir = '../data/SelfDiff_SSD_20251117_threshold1_control'
# output_dir = f"../data/SelfDiff_SSD_{datetime.now().strftime('%Y%m%d')}_control"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Cycle 1

In [ ]:
prev_cycle = 'exp'
cycle = '1'
df_prev = pd.read_csv(os.path.join(ssd_dir, f'SelfDiff_cycle_{prev_cycle}.csv'))

In [ ]:
df_mol = pd.read_csv('../data/processed/SelfDiff_MDmolecules.csv')
id2smiles = df_mol.set_index('MD_ID')['can_SMILES'].to_dict()
df_add = pd.read_csv('../data/MD/selfdiff/gaff2/diffusion_coefficients_adaptive_full.csv')
df_add['can_SMILES'] = df_add['molecule'].map(id2smiles)
df_add['T_K'] = 300
df_add = df_add[~df_add['can_SMILES'].isin(df_prev['can_SMILES'])]

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df_add['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_add_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings_add=[]
for _, row in df_add.iterrows():
    embeddings_add.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings_add = np.array(embeddings_add)

embeddings_roberta_add = embeddings_add[:, :-1]
temperature_feature_add = embeddings_add[:, -1]

In [ ]:
embeddings_roberta_add.shape

In [ ]:
with open(os.path.join(ssd_dir, 'pca_object_cycle_exp.pkl'), 'rb') as f:
        loaded_pca = pickle.load(f)
    
embeddings_roberta_reduced_add = loaded_pca.fit_transform(embeddings_roberta_add)
print("Shape of reduced embeddings:", embeddings_roberta_reduced_add.shape)

# Reshape temperature_feature to make it a 2D column vector
temperature_feature_add = temperature_feature_add.reshape(-1, 1)  # Shape: (2398, 1)

# Concatenate the reduced embeddings with the temperature feature along the columns
combined_embeddings_add = np.concatenate([embeddings_roberta_reduced_add, temperature_feature_add], axis=1)
print("Shape of combined features:", combined_embeddings_add.shape)

In [ ]:
# …later, in a fresh Python session or notebook
best_rf_loaded = joblib.load(os.path.join(ssd_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{prev_cycle}.joblib"))

# Use it exactly the same way
y_pred  = best_rf_loaded.predict(combined_embeddings_add)

In [ ]:
df_add = df_add.rename(columns={'molecule': 'MD_ID', 'D_m2_s': 'Dmd_m2_s'})
Dmd_e9_m2_s=[]
for _, row in df_add.iterrows():
    Dmd_e9_m2_s.append(row.Dmd_m2_s*(10**9))
df_add['Dmd×10-9/m2·s-1']=Dmd_e9_m2_s
df_add['Dpred×10-9/m2·s-1']=y_pred

print('length of df to add', len(df_add))

In [ ]:
df_aug = df_add.sample(n=701, random_state=42)
df_aug['ln(Dmd×10-9/m2·s-1)'] = np.log(df_aug['Dmd×10-9/m2·s-1'])
df_aug['ln(Dpred×10-9/m2·s-1)'] = np.log(df_aug['Dpred×10-9/m2·s-1'])
df_aug['ref']=['MD']*len(df_aug)

df_leftover = df_add.drop(df_aug.index)

print('length of df_aug', len(df_aug))
print('length of df_leftover', len(df_leftover))

df_aug.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_aug.csv'), index=False)
df_leftover.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_leftover.csv'), index=False)

In [ ]:
df = pd.concat([df_prev, df_aug])
df = df[['can_SMILES', 'T_K', 'Dexp×10-9/m2·s-1', 'box', 'window_ps', 'Dmd×10-9/m2·s-1', 'Dpred×10-9/m2·s-1', 'ref']]

# prefer the experimental value; if it’s NaN take the prediction
df['target'] = (df['Dexp×10-9/m2·s-1'].fillna(df['Dpred×10-9/m2·s-1']))

both_present = (
    df['Dexp×10-9/m2·s-1'].notna() &
    df['Dpred×10-9/m2·s-1'].notna()
)
assert not both_present.any(), "Found rows where both Dexp and Dpred are populated!"

df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}.csv'), index=False)

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings = []
for _, row in df.iterrows():
    embeddings.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings = np.array(embeddings)

embeddings_roberta = embeddings[:, :-1]
temperature_feature = embeddings[:, -1]
embeddings_roberta.shape

In [ ]:
embeddings_roberta_reduced = loaded_pca.fit_transform(embeddings_roberta)
embeddings_roberta_reduced.shape

In [ ]:
temperature_feature = temperature_feature.reshape(-1, 1)  # Shape: (2398, 1)
combined_embeddings = np.concatenate([embeddings_roberta_reduced, temperature_feature], axis=1)
print("Shape of combined features:", combined_embeddings.shape)


In [ ]:
embeddings_reduced_path = os.path.join(output_dir, f"embeddings_roberta_reduced_cycle_{cycle}.pkl")
with open(embeddings_reduced_path, "wb") as f:
    pickle.dump({
        "embeddings": embeddings_roberta_reduced,
        "cycle": cycle
    }, f)
print(f"Saved reduced embeddings to {embeddings_reduced_path}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = combined_embeddings
y = df['target']
assert X.shape[0] == len(y), f"Mismatch: {X.shape[0]} embeddings vs {len(y)} labels"

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [ ]:
# Random Forest

from sklearn.model_selection import KFold
import pandas as pd

pipeline_rf_roberta = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=1, criterion='absolute_error'))
])

grid_search_rf_roberta = GridSearchCV(
    pipeline_rf_roberta,
    param_grid = {
        'rf__n_estimators': [10, 50, 100, 200],
        'rf__max_depth': [2, 5, 10],
        'rf__min_samples_split': [2, 5, 10],
        'rf__min_samples_leaf': [1, 2, 4]},
    cv=5,
    verbose=2
)

grid_search_rf_roberta.fit(X_train, y_train)

# Best estimator and predictions
best_rf_roberta = grid_search_rf_roberta.best_estimator_
y_pred_rf_roberta = best_rf_roberta.predict(X)
df['Pred'] = y_pred_rf_roberta
y_train_pred_rf_roberta = best_rf_roberta.predict(X_train)
y_test_pred_rf_roberta = best_rf_roberta.predict(X_test)

# Metrics
metrics_dict = {
    "R2_all": r2_score(y, y_pred_rf_roberta),
    "MAE_all": mean_absolute_error(y, y_pred_rf_roberta),
    "R2_train": r2_score(y_train, y_train_pred_rf_roberta),
    "MAE_train": mean_absolute_error(y_train, y_train_pred_rf_roberta),
    "R2_test": r2_score(y_test, y_test_pred_rf_roberta),
    "MAE_test": mean_absolute_error(y_test, y_test_pred_rf_roberta),
}

print("Best parameters:", grid_search_rf_roberta.best_params_)
print("R²:", metrics_dict["R2_all"])
print("MAE:", metrics_dict["MAE_all"])
print("Train R²:", metrics_dict["R2_train"])
print("Train MAE:", metrics_dict["MAE_train"])
print("Test R²:", metrics_dict["R2_test"])
print("Test MAE:", metrics_dict["MAE_test"])

# Cross-validation predictions for variance per molecule
kf = KFold(n_splits=5, shuffle=True, random_state=1)
all_preds = np.zeros((X.shape[0], kf.get_n_splits()))
fold_indices = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    model_rf = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            random_state=1,
            criterion='absolute_error',
            n_estimators=grid_search_rf_roberta.best_params_['rf__n_estimators'],
            max_depth=grid_search_rf_roberta.best_params_['rf__max_depth'],
            min_samples_split=grid_search_rf_roberta.best_params_['rf__min_samples_split'],
            min_samples_leaf=grid_search_rf_roberta.best_params_['rf__min_samples_leaf'],
        ))
    ])
    model_rf.fit(X[train_idx], y.iloc[train_idx])
    preds = model_rf.predict(X)
    all_preds[:, fold] = preds
    fold_indices.append(test_idx)

# Per-molecule mean and std across CV folds
preds_mean = np.mean(all_preds, axis=1)
preds_std = np.std(all_preds, axis=1)

# Entity labels
if 'Compound_Name' in df.columns:
    entity_labels = df['Compound_Name']
else:
    entity_labels = df.index.astype(str)

# Save per-molecule prediction stats for plotting
df['Pred_CV_Mean'] = preds_mean
df['Pred_CV_Std'] = preds_std

# Save all info needed for plotting
results_df = df.copy()
results_df['y_true'] = y.values

# Mark train/test membership using y_train/y_test indices (which are pandas Index)
results_df['is_train'] = results_df.index.isin(y_train.index)
results_df['is_test'] = results_df.index.isin(y_test.index)

# Save per-molecule results (for parity plot with variance)
results_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_results.csv'), index=False)

# Save held-out test set performance for model performance plot
test_perf_df = pd.DataFrame({
    'cycle': [cycle],
    'R2_test': [metrics_dict["R2_test"]],
    'MAE_test': [metrics_dict["MAE_test"]],
    'n_test': [len(y_test)]
})
test_perf_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_test_performance.csv'), index=False)

# Save cross-validation results for revision
cv_results_rf_roberta = grid_search_rf_roberta.cv_results_
cv_results_df = pd.DataFrame(cv_results_rf_roberta)
cv_results_df.to_csv(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_cv_results_cycle_{cycle}.csv"), index=False)

# Save best parameters and all metrics for revision
with open(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_params_cycle_{cycle}.txt"), "w") as f:
    f.write(f"Timestep: cycle{cycle}\n")
    f.write(f"Best Parameters :\n")
    f.write(str(grid_search_rf_roberta.best_params_) + "\n\n")
    f.write(f"Mean test score for best parameters : {cv_results_rf_roberta['mean_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write(f"Standard deviation of test score : {cv_results_rf_roberta['std_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write("\n")
    for k, v in metrics_dict.items():
        f.write(f"{k} : {v}\n")
    f.write(f"n_train : {len(y_train)}\n")
    f.write(f"n_test : {len(y_test)}\n")

In [ ]:
joblib.dump(best_rf_roberta, os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{cycle}.joblib"))

# Cycle 2

In [ ]:
prev_cycle = '1'
cycle = '2'
df_prev = pd.read_csv(os.path.join(output_dir, f'SelfDiff_cycle_{prev_cycle}.csv'))
df_add = pd.read_csv(os.path.join(output_dir, f'SelfDiff_cycle_{prev_cycle}_leftover.csv'))
df_add = df_add[~df_add['can_SMILES'].isin(df_prev['can_SMILES'])]

print('length of df_prev', len(df_prev))
print('length of df_add', len(df_add))

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df_add['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_add_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings_add=[]
for _, row in df_add.iterrows():
    embeddings_add.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings_add = np.array(embeddings_add)

embeddings_roberta_add = embeddings_add[:, :-1]
temperature_feature_add = embeddings_add[:, -1]

In [ ]:
embeddings_roberta_add.shape

In [ ]:
with open(os.path.join(ssd_dir, 'pca_object_cycle_exp.pkl'), 'rb') as f:
        loaded_pca = pickle.load(f)
    
embeddings_roberta_reduced_add = loaded_pca.fit_transform(embeddings_roberta_add)
print("Shape of reduced embeddings:", embeddings_roberta_reduced_add.shape)

# Reshape temperature_feature to make it a 2D column vector
temperature_feature_add = temperature_feature_add.reshape(-1, 1)  # Shape: (2398, 1)

# Concatenate the reduced embeddings with the temperature feature along the columns
combined_embeddings_add = np.concatenate([embeddings_roberta_reduced_add, temperature_feature_add], axis=1)
print("Shape of combined features:", combined_embeddings_add.shape)

In [ ]:
# …later, in a fresh Python session or notebook
best_rf_loaded = joblib.load(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{prev_cycle}.joblib"))

# Use it exactly the same way
y_pred  = best_rf_loaded.predict(combined_embeddings_add)

In [ ]:
df_add['Dpred×10-9/m2·s-1']=y_pred
df_aug = df_add.sample(n=245, random_state=42)
df_aug['ln(Dmd×10-9/m2·s-1)'] = np.log(df_aug['Dmd×10-9/m2·s-1'])
df_aug['ln(Dpred×10-9/m2·s-1)'] = np.log(df_aug['Dpred×10-9/m2·s-1'])
df_aug['ref']=['MD']*len(df_aug)

df_leftover = df_add.drop(df_aug.index)

print('length of df_aug', len(df_aug))
print('length of df_leftover', len(df_leftover))

df_aug.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_aug.csv'), index=False)
df_leftover.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_leftover.csv'), index=False)

In [ ]:
df = pd.concat([df_prev, df_aug])
df = df[['can_SMILES', 'T_K', 'Dexp×10-9/m2·s-1', 'box', 'window_ps', 'Dmd×10-9/m2·s-1', 'Dpred×10-9/m2·s-1', 'ref']]

# prefer the experimental value; if it’s NaN take the prediction
df['target'] = (df['Dexp×10-9/m2·s-1'].fillna(df['Dpred×10-9/m2·s-1']))

both_present = (
    df['Dexp×10-9/m2·s-1'].notna() &
    df['Dpred×10-9/m2·s-1'].notna()
)
assert not both_present.any(), "Found rows where both Dexp and Dpred are populated!"

df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}.csv'), index=False)

print('length of df', len(df))

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings = []
for _, row in df.iterrows():
    embeddings.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings = np.array(embeddings)

embeddings_roberta = embeddings[:, :-1]
temperature_feature = embeddings[:, -1]
embeddings_roberta.shape

In [ ]:
embeddings_roberta_reduced = loaded_pca.fit_transform(embeddings_roberta)
embeddings_roberta_reduced.shape

In [ ]:
temperature_feature = temperature_feature.reshape(-1, 1)  # Shape: (2398, 1)
combined_embeddings = np.concatenate([embeddings_roberta_reduced, temperature_feature], axis=1)
print("Shape of combined features:", combined_embeddings.shape)


In [ ]:
embeddings_reduced_path = os.path.join(output_dir, f"embeddings_roberta_reduced_cycle_{cycle}.pkl")
with open(embeddings_reduced_path, "wb") as f:
    pickle.dump({
        "embeddings": embeddings_roberta_reduced,
        "cycle": cycle
    }, f)
print(f"Saved reduced embeddings to {embeddings_reduced_path}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = combined_embeddings
y = df['target']
assert X.shape[0] == len(y), f"Mismatch: {X.shape[0]} embeddings vs {len(y)} labels"

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [ ]:
# Random Forest

from sklearn.model_selection import KFold
import pandas as pd

pipeline_rf_roberta = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=1, criterion='absolute_error'))
])

grid_search_rf_roberta = GridSearchCV(
    pipeline_rf_roberta,
    param_grid = {
        'rf__n_estimators': [10, 50, 100, 200],
        'rf__max_depth': [2, 5, 10],
        'rf__min_samples_split': [2, 5, 10],
        'rf__min_samples_leaf': [1, 2, 4]},
    cv=5,
    verbose=2
)

grid_search_rf_roberta.fit(X_train, y_train)

# Best estimator and predictions
best_rf_roberta = grid_search_rf_roberta.best_estimator_
y_pred_rf_roberta = best_rf_roberta.predict(X)
df['Pred'] = y_pred_rf_roberta
y_train_pred_rf_roberta = best_rf_roberta.predict(X_train)
y_test_pred_rf_roberta = best_rf_roberta.predict(X_test)

# Metrics
metrics_dict = {
    "R2_all": r2_score(y, y_pred_rf_roberta),
    "MAE_all": mean_absolute_error(y, y_pred_rf_roberta),
    "R2_train": r2_score(y_train, y_train_pred_rf_roberta),
    "MAE_train": mean_absolute_error(y_train, y_train_pred_rf_roberta),
    "R2_test": r2_score(y_test, y_test_pred_rf_roberta),
    "MAE_test": mean_absolute_error(y_test, y_test_pred_rf_roberta),
}

print("Best parameters:", grid_search_rf_roberta.best_params_)
print("R²:", metrics_dict["R2_all"])
print("MAE:", metrics_dict["MAE_all"])
print("Train R²:", metrics_dict["R2_train"])
print("Train MAE:", metrics_dict["MAE_train"])
print("Test R²:", metrics_dict["R2_test"])
print("Test MAE:", metrics_dict["MAE_test"])

# Cross-validation predictions for variance per molecule
kf = KFold(n_splits=5, shuffle=True, random_state=1)
all_preds = np.zeros((X.shape[0], kf.get_n_splits()))
fold_indices = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    model_rf = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            random_state=1,
            criterion='absolute_error',
            n_estimators=grid_search_rf_roberta.best_params_['rf__n_estimators'],
            max_depth=grid_search_rf_roberta.best_params_['rf__max_depth'],
            min_samples_split=grid_search_rf_roberta.best_params_['rf__min_samples_split'],
            min_samples_leaf=grid_search_rf_roberta.best_params_['rf__min_samples_leaf'],
        ))
    ])
    model_rf.fit(X[train_idx], y.iloc[train_idx])
    preds = model_rf.predict(X)
    all_preds[:, fold] = preds
    fold_indices.append(test_idx)

# Per-molecule mean and std across CV folds
preds_mean = np.mean(all_preds, axis=1)
preds_std = np.std(all_preds, axis=1)

# Entity labels
if 'Compound_Name' in df.columns:
    entity_labels = df['Compound_Name']
else:
    entity_labels = df.index.astype(str)

# Save per-molecule prediction stats for plotting
df['Pred_CV_Mean'] = preds_mean
df['Pred_CV_Std'] = preds_std

# Save all info needed for plotting
results_df = df.copy()
results_df['y_true'] = y.values

# Mark train/test membership using y_train/y_test indices (which are pandas Index)
results_df['is_train'] = results_df.index.isin(y_train.index)
results_df['is_test'] = results_df.index.isin(y_test.index)

# Save per-molecule results (for parity plot with variance)
results_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_results.csv'), index=False)

# Save held-out test set performance for model performance plot
test_perf_df = pd.DataFrame({
    'cycle': [cycle],
    'R2_test': [metrics_dict["R2_test"]],
    'MAE_test': [metrics_dict["MAE_test"]],
    'n_test': [len(y_test)]
})
test_perf_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_test_performance.csv'), index=False)

# Save cross-validation results for revision
cv_results_rf_roberta = grid_search_rf_roberta.cv_results_
cv_results_df = pd.DataFrame(cv_results_rf_roberta)
cv_results_df.to_csv(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_cv_results_cycle_{cycle}.csv"), index=False)

# Save best parameters and all metrics for revision
with open(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_params_cycle_{cycle}.txt"), "w") as f:
    f.write(f"Timestep: cycle{cycle}\n")
    f.write(f"Best Parameters :\n")
    f.write(str(grid_search_rf_roberta.best_params_) + "\n\n")
    f.write(f"Mean test score for best parameters : {cv_results_rf_roberta['mean_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write(f"Standard deviation of test score : {cv_results_rf_roberta['std_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write("\n")
    for k, v in metrics_dict.items():
        f.write(f"{k} : {v}\n")
    f.write(f"n_train : {len(y_train)}\n")
    f.write(f"n_test : {len(y_test)}\n")

In [ ]:
joblib.dump(best_rf_roberta, os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{cycle}.joblib"))

# Cycle 3

In [ ]:
prev_cycle = '2'
cycle = '3'
df_prev = pd.read_csv(os.path.join(output_dir, f'SelfDiff_cycle_{prev_cycle}.csv'))
df_add = pd.read_csv(os.path.join(output_dir, f'SelfDiff_cycle_{prev_cycle}_leftover.csv'))
df_add = df_add[~df_add['can_SMILES'].isin(df_prev['can_SMILES'])]

print('length of df_prev', len(df_prev))
print('length of df_add', len(df_add))

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df_add['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_add_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings_add=[]
for _, row in df_add.iterrows():
    embeddings_add.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings_add = np.array(embeddings_add)

embeddings_roberta_add = embeddings_add[:, :-1]
temperature_feature_add = embeddings_add[:, -1]

In [ ]:
embeddings_roberta_add.shape

In [ ]:
with open(os.path.join(ssd_dir, 'pca_object_cycle_exp.pkl'), 'rb') as f:
        loaded_pca = pickle.load(f)
    
embeddings_roberta_reduced_add = loaded_pca.fit_transform(embeddings_roberta_add)
print("Shape of reduced embeddings:", embeddings_roberta_reduced_add.shape)

# Reshape temperature_feature to make it a 2D column vector
temperature_feature_add = temperature_feature_add.reshape(-1, 1)  # Shape: (2398, 1)

# Concatenate the reduced embeddings with the temperature feature along the columns
combined_embeddings_add = np.concatenate([embeddings_roberta_reduced_add, temperature_feature_add], axis=1)
print("Shape of combined features:", combined_embeddings_add.shape)

In [ ]:
# …later, in a fresh Python session or notebook
best_rf_loaded = joblib.load(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{prev_cycle}.joblib"))

# Use it exactly the same way
y_pred  = best_rf_loaded.predict(combined_embeddings_add)

In [ ]:
df_add['Dpred×10-9/m2·s-1']=y_pred
df_aug = df_add.sample(n=23, random_state=42)
df_aug['ln(Dmd×10-9/m2·s-1)'] = np.log(df_aug['Dmd×10-9/m2·s-1'])
df_aug['ln(Dpred×10-9/m2·s-1)'] = np.log(df_aug['Dpred×10-9/m2·s-1'])
df_aug['ref']=['MD']*len(df_aug)

df_leftover = df_add.drop(df_aug.index)

print('length of df_aug', len(df_aug))
print('length of df_leftover', len(df_leftover))

df_aug.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_aug.csv'), index=False)
df_leftover.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_leftover.csv'), index=False)

In [ ]:
df = pd.concat([df_prev, df_aug])
df = df[['can_SMILES', 'T_K', 'Dexp×10-9/m2·s-1', 'box', 'window_ps', 'Dmd×10-9/m2·s-1', 'Dpred×10-9/m2·s-1', 'ref']]

# prefer the experimental value; if it’s NaN take the prediction
df['target'] = (df['Dexp×10-9/m2·s-1'].fillna(df['Dpred×10-9/m2·s-1']))

both_present = (
    df['Dexp×10-9/m2·s-1'].notna() &
    df['Dpred×10-9/m2·s-1'].notna()
)
assert not both_present.any(), "Found rows where both Dexp and Dpred are populated!"

df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}.csv'), index=False)

print('length of df', len(df))

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings = []
for _, row in df.iterrows():
    embeddings.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings = np.array(embeddings)

embeddings_roberta = embeddings[:, :-1]
temperature_feature = embeddings[:, -1]
embeddings_roberta.shape

In [ ]:
embeddings_roberta_reduced = loaded_pca.fit_transform(embeddings_roberta)
embeddings_roberta_reduced.shape

In [ ]:
temperature_feature = temperature_feature.reshape(-1, 1)  # Shape: (2398, 1)
combined_embeddings = np.concatenate([embeddings_roberta_reduced, temperature_feature], axis=1)
print("Shape of combined features:", combined_embeddings.shape)


In [ ]:
embeddings_reduced_path = os.path.join(output_dir, f"embeddings_roberta_reduced_cycle_{cycle}.pkl")
with open(embeddings_reduced_path, "wb") as f:
    pickle.dump({
        "embeddings": embeddings_roberta_reduced,
        "cycle": cycle
    }, f)
print(f"Saved reduced embeddings to {embeddings_reduced_path}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = combined_embeddings
y = df['target']
assert X.shape[0] == len(y), f"Mismatch: {X.shape[0]} embeddings vs {len(y)} labels"

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [ ]:
# Random Forest

from sklearn.model_selection import KFold
import pandas as pd

pipeline_rf_roberta = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=1, criterion='absolute_error'))
])

grid_search_rf_roberta = GridSearchCV(
    pipeline_rf_roberta,
    param_grid = {
        'rf__n_estimators': [10, 50, 100, 200],
        'rf__max_depth': [2, 5, 10],
        'rf__min_samples_split': [2, 5, 10],
        'rf__min_samples_leaf': [1, 2, 4]},
    cv=5,
    verbose=2
)

grid_search_rf_roberta.fit(X_train, y_train)

# Best estimator and predictions
best_rf_roberta = grid_search_rf_roberta.best_estimator_
y_pred_rf_roberta = best_rf_roberta.predict(X)
df['Pred'] = y_pred_rf_roberta
y_train_pred_rf_roberta = best_rf_roberta.predict(X_train)
y_test_pred_rf_roberta = best_rf_roberta.predict(X_test)

# Metrics
metrics_dict = {
    "R2_all": r2_score(y, y_pred_rf_roberta),
    "MAE_all": mean_absolute_error(y, y_pred_rf_roberta),
    "R2_train": r2_score(y_train, y_train_pred_rf_roberta),
    "MAE_train": mean_absolute_error(y_train, y_train_pred_rf_roberta),
    "R2_test": r2_score(y_test, y_test_pred_rf_roberta),
    "MAE_test": mean_absolute_error(y_test, y_test_pred_rf_roberta),
}

print("Best parameters:", grid_search_rf_roberta.best_params_)
print("R²:", metrics_dict["R2_all"])
print("MAE:", metrics_dict["MAE_all"])
print("Train R²:", metrics_dict["R2_train"])
print("Train MAE:", metrics_dict["MAE_train"])
print("Test R²:", metrics_dict["R2_test"])
print("Test MAE:", metrics_dict["MAE_test"])

# Cross-validation predictions for variance per molecule
kf = KFold(n_splits=5, shuffle=True, random_state=1)
all_preds = np.zeros((X.shape[0], kf.get_n_splits()))
fold_indices = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    model_rf = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            random_state=1,
            criterion='absolute_error',
            n_estimators=grid_search_rf_roberta.best_params_['rf__n_estimators'],
            max_depth=grid_search_rf_roberta.best_params_['rf__max_depth'],
            min_samples_split=grid_search_rf_roberta.best_params_['rf__min_samples_split'],
            min_samples_leaf=grid_search_rf_roberta.best_params_['rf__min_samples_leaf'],
        ))
    ])
    model_rf.fit(X[train_idx], y.iloc[train_idx])
    preds = model_rf.predict(X)
    all_preds[:, fold] = preds
    fold_indices.append(test_idx)

# Per-molecule mean and std across CV folds
preds_mean = np.mean(all_preds, axis=1)
preds_std = np.std(all_preds, axis=1)

# Entity labels
if 'Compound_Name' in df.columns:
    entity_labels = df['Compound_Name']
else:
    entity_labels = df.index.astype(str)

# Save per-molecule prediction stats for plotting
df['Pred_CV_Mean'] = preds_mean
df['Pred_CV_Std'] = preds_std

# Save all info needed for plotting
results_df = df.copy()
results_df['y_true'] = y.values

# Mark train/test membership using y_train/y_test indices (which are pandas Index)
results_df['is_train'] = results_df.index.isin(y_train.index)
results_df['is_test'] = results_df.index.isin(y_test.index)

# Save per-molecule results (for parity plot with variance)
results_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_results.csv'), index=False)

# Save held-out test set performance for model performance plot
test_perf_df = pd.DataFrame({
    'cycle': [cycle],
    'R2_test': [metrics_dict["R2_test"]],
    'MAE_test': [metrics_dict["MAE_test"]],
    'n_test': [len(y_test)]
})
test_perf_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_test_performance.csv'), index=False)

# Save cross-validation results for revision
cv_results_rf_roberta = grid_search_rf_roberta.cv_results_
cv_results_df = pd.DataFrame(cv_results_rf_roberta)
cv_results_df.to_csv(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_cv_results_cycle_{cycle}.csv"), index=False)

# Save best parameters and all metrics for revision
with open(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_params_cycle_{cycle}.txt"), "w") as f:
    f.write(f"Timestep: cycle{cycle}\n")
    f.write(f"Best Parameters :\n")
    f.write(str(grid_search_rf_roberta.best_params_) + "\n\n")
    f.write(f"Mean test score for best parameters : {cv_results_rf_roberta['mean_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write(f"Standard deviation of test score : {cv_results_rf_roberta['std_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write("\n")
    for k, v in metrics_dict.items():
        f.write(f"{k} : {v}\n")
    f.write(f"n_train : {len(y_train)}\n")
    f.write(f"n_test : {len(y_test)}\n")

In [ ]:
joblib.dump(best_rf_roberta, os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{cycle}.joblib"))

# Cycle 4

In [ ]:
prev_cycle = '3'
cycle = '4'
df_prev = pd.read_csv(os.path.join(output_dir, f'SelfDiff_cycle_{prev_cycle}.csv'))
df_add = pd.read_csv(os.path.join(output_dir, f'SelfDiff_cycle_{prev_cycle}_leftover.csv'))
df_add = df_add[~df_add['can_SMILES'].isin(df_prev['can_SMILES'])]

print('length of df_prev', len(df_prev))
print('length of df_add', len(df_add))

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df_add['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_add_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings_add=[]
for _, row in df_add.iterrows():
    embeddings_add.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings_add = np.array(embeddings_add)

embeddings_roberta_add = embeddings_add[:, :-1]
temperature_feature_add = embeddings_add[:, -1]

In [ ]:
embeddings_roberta_add.shape

In [ ]:
with open(os.path.join(ssd_dir, 'pca_object_cycle_exp.pkl'), 'rb') as f:
        loaded_pca = pickle.load(f)
    
embeddings_roberta_reduced_add = loaded_pca.fit_transform(embeddings_roberta_add)
print("Shape of reduced embeddings:", embeddings_roberta_reduced_add.shape)

# Reshape temperature_feature to make it a 2D column vector
temperature_feature_add = temperature_feature_add.reshape(-1, 1)  # Shape: (2398, 1)

# Concatenate the reduced embeddings with the temperature feature along the columns
combined_embeddings_add = np.concatenate([embeddings_roberta_reduced_add, temperature_feature_add], axis=1)
print("Shape of combined features:", combined_embeddings_add.shape)

In [ ]:
# …later, in a fresh Python session or notebook
best_rf_loaded = joblib.load(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{prev_cycle}.joblib"))

# Use it exactly the same way
y_pred  = best_rf_loaded.predict(combined_embeddings_add)

In [ ]:
df_add['Dpred×10-9/m2·s-1']=y_pred
df_aug = df_add.sample(n=4, random_state=42)
df_aug['ln(Dmd×10-9/m2·s-1)'] = np.log(df_aug['Dmd×10-9/m2·s-1'])
df_aug['ln(Dpred×10-9/m2·s-1)'] = np.log(df_aug['Dpred×10-9/m2·s-1'])
df_aug['ref']=['MD']*len(df_aug)

df_leftover = df_add.drop(df_aug.index)

print('length of df_aug', len(df_aug))
print('length of df_leftover', len(df_leftover))

df_aug.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_aug.csv'), index=False)
df_leftover.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_leftover.csv'), index=False)

In [ ]:
df = pd.concat([df_prev, df_aug])
df = df[['can_SMILES', 'T_K', 'Dexp×10-9/m2·s-1', 'box', 'window_ps', 'Dmd×10-9/m2·s-1', 'Dpred×10-9/m2·s-1', 'ref']]

# prefer the experimental value; if it’s NaN take the prediction
df['target'] = (df['Dexp×10-9/m2·s-1'].fillna(df['Dpred×10-9/m2·s-1']))

both_present = (
    df['Dexp×10-9/m2·s-1'].notna() &
    df['Dpred×10-9/m2·s-1'].notna()
)
assert not both_present.any(), "Found rows where both Dexp and Dpred are populated!"

df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}.csv'), index=False)

print('length of df', len(df))

In [ ]:
# Unique SMILES to embed
unique_smiles = (
    df['can_SMILES']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Output directory for parquet parts (each batch → one file)
OUT_DIR = Path(os.path.join(output_dir, f"embedded_SMILES_parts_df_cycle_{cycle}"))
OUT_DIR.mkdir(exist_ok=True)

def already_done_set():
    done = set()
    parts = sorted(OUT_DIR.glob("part_*.parquet"))
    for p in parts:
        # read only the 'can_SMILES' column to keep memory tiny
        tbl = pq.read_table(p, columns=["can_SMILES"])
        done.update(tbl.column("can_SMILES").to_pylist())
    return done

done = already_done_set()
todo = [s for s in unique_smiles if s not in done]
print(f"Total: {len(unique_smiles)} | already saved: {len(done)} | remaining: {len(todo)}")

BATCH_SIZE   = 16         # lower to 8 or 4 if memory is tight
FLUSH_SLEEP  = 0.0        # set e.g. 0.1s if your laptop gets hot
EMB_COLS     = None       # determined after first batch

def save_batch(smiles_batch, emb_np):
    global EMB_COLS
    if EMB_COLS is None:
        EMB_COLS = [f"e{i}" for i in range(emb_np.shape[1])]
    df_emb = pd.DataFrame(emb_np, columns=EMB_COLS)
    df_emb.insert(0, "can_SMILES", smiles_batch)
    # unique filename per batch
    ts = int(time.time() * 1000)
    part_path = OUT_DIR / f"part_{ts}_{len(smiles_batch)}.parquet"
    df_emb.to_parquet(part_path, index=False, compression="zstd")

processed = 0
for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    # Use your function exactly as-is
    emb = Create_embedding(batch).detach().cpu().numpy()
    save_batch(batch, emb)
    processed += len(batch)
    if FLUSH_SLEEP: time.sleep(FLUSH_SLEEP)
    # free transient memory
    del emb; gc.collect()
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Progress: {processed}/{len(todo)} newly saved")

print("Finished batch writing to", OUT_DIR)

# Read the directory of parquet parts as one dataset
parts = sorted(glob.glob(str(OUT_DIR / "part_*.parquet")))
if not parts:
    raise FileNotFoundError("No parquet parts found. Run the batch cell first.")

# Concatenate efficiently
tables = [pq.read_table(p) for p in parts]
full = pa.concat_tables(tables)
df_emb_all = full.to_pandas()  # single dataframe with all embeddings
df_emb_all.shape, df_emb_all.head()

# Map can_SMILES -> embedding vector (numpy)
embedded_SMILES = {
    s: v for s, v in zip(
        df_emb_all["can_SMILES"].tolist(),
        df_emb_all.iloc[:, 1:].to_numpy()
    )
}
print ('length of embedded_SMILES', len(embedded_SMILES))

In [ ]:
embeddings = []
for _, row in df.iterrows():
    embeddings.append(list(embedded_SMILES[row['can_SMILES']])+[row['T_K']])
embeddings = np.array(embeddings)

embeddings_roberta = embeddings[:, :-1]
temperature_feature = embeddings[:, -1]
embeddings_roberta.shape

In [ ]:
embeddings_roberta_reduced = loaded_pca.fit_transform(embeddings_roberta)
embeddings_roberta_reduced.shape

In [ ]:
temperature_feature = temperature_feature.reshape(-1, 1)  # Shape: (2398, 1)
combined_embeddings = np.concatenate([embeddings_roberta_reduced, temperature_feature], axis=1)
print("Shape of combined features:", combined_embeddings.shape)


In [ ]:
embeddings_reduced_path = os.path.join(output_dir, f"embeddings_roberta_reduced_cycle_{cycle}.pkl")
with open(embeddings_reduced_path, "wb") as f:
    pickle.dump({
        "embeddings": embeddings_roberta_reduced,
        "cycle": cycle
    }, f)
print(f"Saved reduced embeddings to {embeddings_reduced_path}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = combined_embeddings
y = df['target']
assert X.shape[0] == len(y), f"Mismatch: {X.shape[0]} embeddings vs {len(y)} labels"

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [ ]:
# Random Forest

from sklearn.model_selection import KFold
import pandas as pd

pipeline_rf_roberta = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=1, criterion='absolute_error'))
])

grid_search_rf_roberta = GridSearchCV(
    pipeline_rf_roberta,
    param_grid = {
        'rf__n_estimators': [10, 50, 100, 200],
        'rf__max_depth': [2, 5, 10],
        'rf__min_samples_split': [2, 5, 10],
        'rf__min_samples_leaf': [1, 2, 4]},
    cv=5,
    verbose=2
)

grid_search_rf_roberta.fit(X_train, y_train)

# Best estimator and predictions
best_rf_roberta = grid_search_rf_roberta.best_estimator_
y_pred_rf_roberta = best_rf_roberta.predict(X)
df['Pred'] = y_pred_rf_roberta
y_train_pred_rf_roberta = best_rf_roberta.predict(X_train)
y_test_pred_rf_roberta = best_rf_roberta.predict(X_test)

# Metrics
metrics_dict = {
    "R2_all": r2_score(y, y_pred_rf_roberta),
    "MAE_all": mean_absolute_error(y, y_pred_rf_roberta),
    "R2_train": r2_score(y_train, y_train_pred_rf_roberta),
    "MAE_train": mean_absolute_error(y_train, y_train_pred_rf_roberta),
    "R2_test": r2_score(y_test, y_test_pred_rf_roberta),
    "MAE_test": mean_absolute_error(y_test, y_test_pred_rf_roberta),
}

print("Best parameters:", grid_search_rf_roberta.best_params_)
print("R²:", metrics_dict["R2_all"])
print("MAE:", metrics_dict["MAE_all"])
print("Train R²:", metrics_dict["R2_train"])
print("Train MAE:", metrics_dict["MAE_train"])
print("Test R²:", metrics_dict["R2_test"])
print("Test MAE:", metrics_dict["MAE_test"])

# Cross-validation predictions for variance per molecule
kf = KFold(n_splits=5, shuffle=True, random_state=1)
all_preds = np.zeros((X.shape[0], kf.get_n_splits()))
fold_indices = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    model_rf = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            random_state=1,
            criterion='absolute_error',
            n_estimators=grid_search_rf_roberta.best_params_['rf__n_estimators'],
            max_depth=grid_search_rf_roberta.best_params_['rf__max_depth'],
            min_samples_split=grid_search_rf_roberta.best_params_['rf__min_samples_split'],
            min_samples_leaf=grid_search_rf_roberta.best_params_['rf__min_samples_leaf'],
        ))
    ])
    model_rf.fit(X[train_idx], y.iloc[train_idx])
    preds = model_rf.predict(X)
    all_preds[:, fold] = preds
    fold_indices.append(test_idx)

# Per-molecule mean and std across CV folds
preds_mean = np.mean(all_preds, axis=1)
preds_std = np.std(all_preds, axis=1)

# Entity labels
if 'Compound_Name' in df.columns:
    entity_labels = df['Compound_Name']
else:
    entity_labels = df.index.astype(str)

# Save per-molecule prediction stats for plotting
df['Pred_CV_Mean'] = preds_mean
df['Pred_CV_Std'] = preds_std

# Save all info needed for plotting
results_df = df.copy()
results_df['y_true'] = y.values

# Mark train/test membership using y_train/y_test indices (which are pandas Index)
results_df['is_train'] = results_df.index.isin(y_train.index)
results_df['is_test'] = results_df.index.isin(y_test.index)

# Save per-molecule results (for parity plot with variance)
results_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_results.csv'), index=False)

# Save held-out test set performance for model performance plot
test_perf_df = pd.DataFrame({
    'cycle': [cycle],
    'R2_test': [metrics_dict["R2_test"]],
    'MAE_test': [metrics_dict["MAE_test"]],
    'n_test': [len(y_test)]
})
test_perf_df.to_csv(os.path.join(output_dir, f'SelfDiff_cycle_{cycle}_test_performance.csv'), index=False)

# Save cross-validation results for revision
cv_results_rf_roberta = grid_search_rf_roberta.cv_results_
cv_results_df = pd.DataFrame(cv_results_rf_roberta)
cv_results_df.to_csv(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_cv_results_cycle_{cycle}.csv"), index=False)

# Save best parameters and all metrics for revision
with open(os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_params_cycle_{cycle}.txt"), "w") as f:
    f.write(f"Timestep: cycle{cycle}\n")
    f.write(f"Best Parameters :\n")
    f.write(str(grid_search_rf_roberta.best_params_) + "\n\n")
    f.write(f"Mean test score for best parameters : {cv_results_rf_roberta['mean_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write(f"Standard deviation of test score : {cv_results_rf_roberta['std_test_score'][grid_search_rf_roberta.best_index_]}\n")
    f.write("\n")
    for k, v in metrics_dict.items():
        f.write(f"{k} : {v}\n")
    f.write(f"n_train : {len(y_train)}\n")
    f.write(f"n_test : {len(y_test)}\n")

In [ ]:
joblib.dump(best_rf_roberta, os.path.join(output_dir, f"SelfDiff_RT_rf_roberta_best_cycle_{cycle}.joblib"))